# Part 2: Multi-source KB + MAI Grounding MCP

Create a fresh KB that combines the restored search indexes with the MAI Grounding MCP server. This uses the current `mcpServer` pattern: low reasoning effort plus a chat model.

## 1 - Setup

In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

import requests
from azure.identity import AzureCliCredential
from dotenv import dotenv_values

API_VERSION = "2026-05-01-preview"
TENANT_ID = "72f988bf-86f1-41af-91ab-2d7cd011db47"
ENV_PATH = Path(".").resolve() / ".env"
file_env = dotenv_values(ENV_PATH) if ENV_PATH.exists() else {}
env = {**file_env, **os.environ}


def require_any(*names: str) -> str:
    for name in names:
        value = env.get(name)
        if value:
            return value.strip()
    raise KeyError(f"Set one of these in {ENV_PATH}: {', '.join(names)}")


ENDPOINT = require_any("AZURE_SEARCH_SERVICE_ENDPOINT", "MAGOTTEI_EASTUS2EUAP_ENDPOINT").rstrip("/")
SEARCH_AUTH_MODE = env.get("AZURE_SEARCH_AUTH_MODE", "aad").strip().lower()
RAW_CHAT_ENDPOINT = env.get("AZURE_OPENAI_ENDPOINT", "").rstrip("/")

def normalize_chat_endpoint(endpoint: str) -> str:
    if ".openai.azure.com" in endpoint:
        resource_name = endpoint.split("//", 1)[1].split(".", 1)[0]
        return f"https://{resource_name}.services.ai.azure.com"
    return endpoint


CHAT_ENDPOINT = normalize_chat_endpoint(RAW_CHAT_ENDPOINT)
CHAT_KEY = env.get("AZURE_OPENAI_KEY", "")
CHAT_DEPLOYMENT = env.get("AZURE_OPENAI_CHATGPT_DEPLOYMENT", "gpt-5.4-mini")
CHAT_MODEL = env.get("AZURE_OPENAI_CHATGPT_MODEL_NAME", CHAT_DEPLOYMENT)
HRDOCS_INDEX = "hrdocs"
HEALTHDOCS_INDEX = "healthdocs"
stamp = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")


def api_url(path: str) -> str:
    sep = "&" if "?" in path else "?"
    return f"{ENDPOINT}{path}{sep}api-version={API_VERSION}"


search_credential = AzureCliCredential(tenant_id=TENANT_ID)


def get_search_bearer_token() -> str:
    return search_credential.get_token("https://search.azure.com/.default").token


session = requests.Session()
if SEARCH_AUTH_MODE == "api-key":
    session.headers.update({"api-key": require_any("MAGOTTEI_EASTUS2EUAP_KEY", "AZURE_SEARCH_ADMIN_KEY"), "Accept": "application/json"})
else:
    session.headers.update({"Authorization": f"Bearer {get_search_bearer_token()}", "Accept": "application/json"})


def pp(response: requests.Response, max_chars: int = 5000):
    print(f"{response.status_code} {response.reason}")
    try:
        print(json.dumps(response.json(), indent=2)[:max_chars])
    except ValueError:
        print(response.text[:max_chars])


def expect_success(response: requests.Response, *allowed: int):
    if response.status_code not in allowed:
        raise RuntimeError(f"Expected HTTP {allowed}, got {response.status_code}: {response.text[:1000]}")
    return response


class KnowledgeBaseClient:
    def __init__(self, session: requests.Session):
        self.session = session

    def retrieve(self, knowledge_base_name: str, body: dict, *, headers: dict | None = None, timeout: int = 180):
        return self.session.post(api_url(f"/knowledgebases('{knowledge_base_name}')/retrieve"), json=body, headers=headers or {}, timeout=timeout)


kb_client = KnowledgeBaseClient(session)


def create_search_index_source(name: str, index_name: str, description: str):
    body = {
        "name": name,
        "description": description,
        "kind": "searchIndex",
        "searchIndexParameters": {
            "searchIndexName": index_name,
            "sourceDataFields": [{"name": "uid"}, {"name": "snippet_parent_id"}, {"name": "blob_path"}, {"name": "snippet"}],
            "searchFields": [{"name": "snippet"}],
            "semanticConfigurationName": "semantic-configuration",
        },
    }
    r = session.put(api_url(f"/knowledgesources('{name}')"), json=body, headers={"Prefer": "return=representation"})
    pp(r)
    expect_success(r, 200, 201)
    return body


def azure_openai_model():
    if not CHAT_ENDPOINT or not CHAT_KEY:
        raise RuntimeError("Set AZURE_OPENAI_ENDPOINT and AZURE_OPENAI_KEY in .env for KBs that use chat models.")
    return {"kind": "azureOpenAI", "azureOpenAIParameters": {"resourceUri": CHAT_ENDPOINT + "/", "apiKey": CHAT_KEY, "deploymentId": CHAT_DEPLOYMENT, "modelName": CHAT_MODEL}}


def create_knowledge_base(name: str, description: str, source_names: list[str], *, output_mode: str = "extractiveData", reasoning: str = "low", include_model: bool = False, instructions: str | None = None):
    body = {"name": name, "description": description, "outputMode": output_mode, "retrievalReasoningEffort": {"kind": reasoning}, "knowledgeSources": [{"name": n} for n in source_names]}
    if include_model:
        body["models"] = [azure_openai_model()]
    if instructions:
        body["retrievalInstructions"] = instructions
    r = session.put(api_url(f"/knowledgebases('{name}')"), json=body, headers={"Prefer": "return=representation"})
    pp(r)
    expect_success(r, 200, 201)
    return body


def search_index_params(source_names: list[str], *, always_query: bool = True) -> list[dict]:
    return [{"kind": "searchIndex", "knowledgeSourceName": n, "includeReferences": True, "includeReferenceSourceData": True, "alwaysQuerySource": always_query} for n in source_names]


def print_copilot_cli_sidequest(knowledge_base_name: str):
    mcp_url = f"{ENDPOINT}/knowledgebases/{knowledge_base_name}/mcp?api-version={API_VERSION}"
    if SEARCH_AUTH_MODE == "api-key":
        service_key = require_any("MAGOTTEI_EASTUS2EUAP_KEY", "AZURE_SEARCH_ADMIN_KEY")
        headers = {"api-key": service_key}
        print("Service key for the MCP api-key header:")
        print(service_key)
    else:
        bearer = get_search_bearer_token()
        headers = {"Authorization": f"Bearer {bearer}"}
        print("Bearer token for the MCP Authorization header (short-lived):")
        print(bearer)
    config = {"mcpServers": {f"lab532-{knowledge_base_name}": {"type": "http", "url": mcp_url, "headers": headers}}}
    print("Open notebooks/copilot-cli-sidequest.md for the sidequest steps.")
    print("KB MCP URL:")
    print(mcp_url)
    print("MCP config snippet:")
    print(json.dumps(config, indent=2))


def cleanup_resources(*, kb_names: list[str], ks_names: list[str]):
    for kb_name in kb_names:
        r = session.delete(api_url(f"/knowledgebases('{kb_name}')"))
        print(f"Delete KB {kb_name}: {r.status_code} {r.reason}")
    for ks_name in ks_names:
        r = session.delete(api_url(f"/knowledgesources('{ks_name}')"))
        print(f"Delete KS {ks_name}: {r.status_code} {r.reason}")


print(f"Search endpoint : {ENDPOINT}")
print(f"Search auth     : {SEARCH_AUTH_MODE}")
print(f"API version     : {API_VERSION}")
print(f"Timestamp stamp : {stamp}")
MAI_ENDPOINT = env.get("MAI_GROUNDING_ENDPOINT", "https://api.microsoft.ai/v3/mcp").rstrip("/")
MAI_KEY = require_any("MAI_GROUNDING_KEY")
MAI_KEY_HEADER = env.get("MAI_GROUNDING_KEY_HEADER", "x-apikey")
MAI_TOOL = env.get("MAI_GROUNDING_TOOL", "web")
print(f"MAI endpoint    : {MAI_ENDPOINT}")
print(f"MAI tool        : {MAI_TOOL}")

## 2 - Direct MCP probe

In [ ]:
mai_session = requests.Session()
mai_session.headers.update({MAI_KEY_HEADER: MAI_KEY, "Accept": "application/json, text/event-stream"})
init_response = mai_session.post(MAI_ENDPOINT, json={"jsonrpc":"2.0","id":1,"method":"initialize","params":{"protocolVersion":"2024-11-05","capabilities":{},"clientInfo":{"name":"lab532-notebook","version":"1.0"}}}, headers={"Content-Type":"application/json"}, timeout=60)
pp(init_response)
expect_success(init_response, 200)
mcp_session_id = init_response.headers.get("mcp-session-id", "")
extra_headers = {"mcp-session-id": mcp_session_id} if mcp_session_id else {}

In [ ]:
tool_response = mai_session.post(MAI_ENDPOINT, json={"jsonrpc":"2.0","id":2,"method":"tools/call","params":{"name":MAI_TOOL,"arguments":{"query":"Azure AI Search knowledge bases 2026 preview"}}}, headers={"Content-Type":"application/json", **extra_headers}, timeout=90)
pp(tool_response)
expect_success(tool_response, 200)

## 3 - Create restored-index and MAI MCP knowledge sources

In [ ]:
HR_KS_NAME = f"lab532-hrdocs-{stamp}"
HEALTH_KS_NAME = f"lab532-healthdocs-{stamp}"
MAI_KS_NAME = f"lab532-mai-grounding-{stamp}"
KS_NAMES = [HR_KS_NAME, HEALTH_KS_NAME, MAI_KS_NAME]
create_search_index_source(HR_KS_NAME, HRDOCS_INDEX, "LAB532 HR documents")
create_search_index_source(HEALTH_KS_NAME, HEALTHDOCS_INDEX, "LAB532 health benefits documents")
mai_ks_body = {"name": MAI_KS_NAME, "kind": "mcpServer", "description": "LAB532 MAI Grounding MCP knowledge source", "mcpServerParameters": {"serverURL": MAI_ENDPOINT, "authentication": {"kind": "storedHeaders", "storedHeadersParameters": {"headers": {MAI_KEY_HEADER: MAI_KEY}}}, "tools": [{"name": MAI_TOOL, "outputParsing": {"kind": "auto"}}]}}
r = session.put(api_url(f"/knowledgesources('{MAI_KS_NAME}')"), json=mai_ks_body, headers={"Prefer":"return=representation"})
pp(r)
expect_success(r, 200, 201)

## 4 - Create the KB

In [ ]:
KB_NAME = f"lab532-mai-grounding-kb-{stamp}"
create_knowledge_base(KB_NAME, "LAB532 KB combining restored indexes and MAI Grounding MCP", KS_NAMES, output_mode="answerSynthesis", reasoning="low", include_model=True, instructions="Use the restored HR and health indexes for company policy questions. Use MAI Grounding for current public web context.")

## 5 - `kb_client.retrieve`: combine company docs and web grounding

In [ ]:
retrieve_body = {"includeActivity": True, "messages": [{"role":"user", "content":[{"type":"text", "text":"Answer with citations: what employee health benefits are described in the company docs, and what is Azure AI Search knowledge base preview?"}]}], "knowledgeSourceParams": [*search_index_params([HR_KS_NAME, HEALTH_KS_NAME]), {"kind":"mcpServer", "knowledgeSourceName":MAI_KS_NAME, "includeReferences":True, "includeReferenceSourceData":True}], "maxRuntimeInSeconds": 120}
r = kb_client.retrieve(KB_NAME, retrieve_body, timeout=180)
pp(r)
expect_success(r, 200)

## 6 - Copilot CLI sidequest checkpoint

Use this KB as a separate MCP server in the [Copilot CLI sidequest](copilot-cli-sidequest.md).

In [ ]:
print_copilot_cli_sidequest(KB_NAME)

## 7 - Cleanup

In [ ]:
cleanup_resources(kb_names=[KB_NAME], ks_names=KS_NAMES)

Next: [Part 3: Add Fabric IQ](part3-fabric-iq-to-kb.ipynb).